In [1]:
# ============================================================
# CELL 0 — BOOTSTRAP  (identical across all pipeline notebooks)
# Finds the repo root (the folder containing .env), puts src/ on the
# import path, and loads shared config + download-log helpers.
# Machine-agnostic: paths come from .env via config.py, never hardcoded.
# ============================================================
import sys                                     # to modify the module search path at runtime
from pathlib import Path                        # portable path handling across Mac/PC

# Walk up from the current working directory until we find the folder that
# contains '.env' — that folder is the repo root. This is what lets the same
# notebook run on any machine without editing paths.
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists())
sys.path.insert(0, str(_root / 'src'))          # make 'import config' / 'import download_log' resolve

from config import *                            # PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, CURRENT_YEAR, ...
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime                   # for any runtime date handling
import pandas as pd                             # primary data-handling library

log = load_log()                                # load the download-log ledger (created if absent)

# --- Verify the bootstrap resolved correctly before proceeding ---
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("CURRENT_YEAR :", CURRENT_YEAR)

PROJECT_ROOT : C:\Users\mjbou\governance-framework
RAW_DIR      : C:\Users\mjbou\governance-framework\data\raw
PROCESSED_DIR: C:\Users\mjbou\governance-framework\data\processed
CURRENT_YEAR : 2026


# Notebook 37 — IMF AREAER De Facto Exchange-Rate-Regime Pipeline

**Concept 8 (macro policy framework) — primary tier-1 source.**

Reads the hand-transcribed source CSV `data/raw/areaer_defacto_regime.csv`
(IMF *Annual Report 2025*, Appendix II.9 — "De Facto Classification of Exchange
Rate Arrangements, as of April 30, 2025"), harmonizes to ISO3, encodes the
exchange-rate-arrangement flexibility ordinal + IMF grouping, runs structural
integrity guards, and writes `data/processed/areaer_er_clean.csv`.

**Why a transcribed source, not a scraper:** the AREAER Online database is
paywalled and the "borderless matrix" PDF has no reliable automated extraction.
The classification is therefore maintained as a hand-transcribed CSV, validated
at transcription time against the PDF's own per-category and per-column country
counts (checksums — all pass for the 2025 vintage).

**Annual refresh = edit the source CSV only** (re-transcribe reclassified
countries, bump `areaer_as_of`); no code changes. See `instructions_data_maintenance.md`.

**Fields carried from source:** `areaer_arrangement` (10-way regime),
`areaer_mpf` (monetary-policy framework, incl. inflation-targeting flag),
`areaer_anchor_currency`, `areaer_reclassified` (regime-change recency), `areaer_as_of`.

In [12]:
# ============================================================
# CELL 2 — CONFIG & METHODOLOGY LOOKUPS
# Defines source_id, input/output paths, and the FIXED methodology
# lookups (arrangement -> flexibility ordinal + IMF group) plus the
# ISO3 override map. These lookups are the ONLY constants in this
# pipeline; all data — including the as-of date — is read from the CSV,
# so there is NO per-update hardcoding here (no VDEM_VERSION-style edits).
# ============================================================

import os   # path joining, consistent with config.py's style

# --- Source identifier for the download-log / source-registry -------------
# Distinct from the FARI pipeline's 'IMF_AREAER' entry: this is the de-facto
# exchange-rate-REGIME product, a different AREAER dataset.
# ⚠️ MANUAL-MAINTENANCE CONSTANT (set once, never changes with data updates).
SOURCE_ID = "IMF_AREAER_ERREGIME"   # provisional — Cell output confirms no collision

# --- Input (hand-maintained source) and output (clean) file paths ---------
INPUT_CSV  = os.path.join(RAW_DIR,       "areaer_defacto_regime.csv")   # transcribed source
OUTPUT_CSV = os.path.join(PROCESSED_DIR, "areaer_er_clean.csv")         # pipeline output

# --- METHODOLOGY LOOKUP 1: arrangement -> flexibility ordinal (1..10) ------
# Order follows the IMF AREAER matrix row order (most fixed = 1 -> most flexible = 10).
# ⚠️ MANUAL-MAINTENANCE CONSTANT: changes ONLY if the IMF revises its 10-category
#    taxonomy (rare); documented in instructions_data_maintenance.md. NOT a per-update edit.
# CAVEATS for the metric pass (do not silently treat this as a clean cardinal scale):
#   (a) 'Other managed arrangement' (8) is the IMF's RESIDUAL bucket, not a true
#       flexibility rank — distressed managers (Venezuela, Syria, Iran) sit here.
#   (b) intra-soft-peg order (3..7) follows matrix order, not a strict flexibility ranking.
ARRANGEMENT_ORDINAL = {
    "No separate legal tender":        1,
    "Currency board":                  2,
    "Conventional peg":                3,
    "Stabilized arrangement":          4,
    "Crawling peg":                    5,
    "Crawl-like arrangement":          6,
    "Pegged within horizontal bands":  7,
    "Other managed arrangement":       8,   # residual — see caveat (a)
    "Floating":                        9,
    "Free floating":                  10,
}

# --- METHODOLOGY LOOKUP 2: arrangement -> IMF 4-way group ------------------
# Matches the IMF's own grouping stated in the Appendix II.9 preamble.
# ⚠️ MANUAL-MAINTENANCE CONSTANT (same maintenance note as Lookup 1).
ARRANGEMENT_GROUP = {
    "No separate legal tender":        "hard_peg",
    "Currency board":                  "hard_peg",
    "Conventional peg":                "soft_peg",
    "Stabilized arrangement":          "soft_peg",
    "Crawling peg":                    "soft_peg",
    "Crawl-like arrangement":          "soft_peg",
    "Pegged within horizontal bands":  "soft_peg",
    "Other managed arrangement":       "other_managed",   # residual
    "Floating":                        "floating",
    "Free floating":                   "floating",
}

# --- ISO3 OVERRIDE MAP: source country_name -> ISO3 -----------------------
# pycountry resolves most names in Cell 5; this handles the ones it can't
# (SARs, unions, ASCII-stripped diacritics, and Kosovo which has no official ISO3).
# ⚠️ MANUAL-MAINTENANCE CONSTANT: extend ONLY if a NEW oddly-named jurisdiction
#    appears — Cell 5 prints any unmatched name so you know exactly what to add here.
#    Documented in instructions_data_maintenance.md.
ISO3_OVERRIDES = {
    "Hong Kong SAR":                    "HKG",
    "Macao SAR":                        "MAC",
    "Kosovo":                           "XKX",   # no official ISO 3166 alpha-3; XKX is the common user code
    "Turkiye":                          "TUR",
    "Sao Tome and Principe":            "STP",
    "Cote d'Ivoire":                    "CIV",
    "Republic of Congo":                "COG",
    "Democratic Republic of the Congo": "COD",
    "Kyrgyz Republic":                  "KGZ",
    "Lao P.D.R.":                       "LAO",
    "Slovak Republic":                  "SVK",
    "Korea":                            "KOR",
    "Russia":                           "RUS",
    "The Bahamas":                      "BHS",
    "The Gambia":                       "GMB",
    "The Netherlands":                  "NLD",
    "St. Kitts and Nevis":              "KNA",
    "St. Lucia":                        "LCA",
    "St. Vincent and the Grenadines":   "VCT",
    "Curacao and Sint Maarten":         "CUW",   # combined AREAER entity -> Curacao (out of framework scope anyway)
    "Micronesia":                       "FSM",   # pycountry canonical is "Micronesia, Federated States of"; bare "Micronesia" misses exact match
    "Vietnam":                          "VNM",   # pycountry canonical name is "Viet Nam"; exact match misses
}  # combined AREAER entity -> Curacao (out of framework scope anyway)

# --- Confirm setup + surface existing source_ids (idempotent, no assert) ---
existing_ids = list(log["source_id"])
print("Input CSV exists :", os.path.exists(INPUT_CSV), "->", INPUT_CSV)
print("Output target    :", OUTPUT_CSV)
print("Arrangement maps :", len(ARRANGEMENT_ORDINAL), "ordinals /", len(ARRANGEMENT_GROUP), "groups")
print("ISO3 overrides   :", len(ISO3_OVERRIDES))
print("Existing source_ids:", existing_ids)
print(f"'{SOURCE_ID}' already in log:", SOURCE_ID in existing_ids,
      "(expect False on first run, True on re-runs)")

Input CSV exists : True -> C:\Users\mjbou\governance-framework\data\raw\areaer_defacto_regime.csv
Output target    : C:\Users\mjbou\governance-framework\data\processed\areaer_er_clean.csv
Arrangement maps : 10 ordinals / 10 groups
ISO3 overrides   : 22
Existing source_ids: ['WGI', 'WJP', 'FH_FIW', 'TI_CPI', 'IMF_SPI', 'UCDP', 'FSI', 'FRASER_REG', 'FRASER_LEGAL', 'VDEM', 'POWELL_THYNE', 'UNODC_HOMICIDE', 'IRENA_CAPACITY', 'WB_WDI', 'YALE_EPI', 'DPI', 'CIVICUS', 'GPI', 'PTS', 'OBS', 'ND_GAIN', 'BCI', 'HANSON_SIGMAN', 'CCP', 'PEI', 'KOF_TRADE', 'WB_INFORMAL', 'ROMELLI_CBI', 'POLITY5', 'NELDA', 'IDEA_PARTIP', 'PEW_GRI', 'IMF_FISCAL_RULES', 'IMF_IMAPP', 'WB_CARBON', 'CLIMATE_LAWS', 'ODIN', 'TI_POLFINANCE', 'RTI_RATING', 'IMF_AREAER', 'CHINN_ITO', 'PEFA', 'OECD_TFI', 'FATF', 'CPJ']
'IMF_AREAER_ERREGIME' already in log: False (expect False on first run, True on re-runs)


In [13]:
# ============================================================
# CELL 3 — LOAD THE HAND-TRANSCRIBED SOURCE CSV
# Reads data/raw/areaer_defacto_regime.csv as all-strings (to preserve the
# as-of date and 'YYYY-MM' reclassification markers exactly), confirms the
# expected schema, and shows shape + a sample. No transformation happens here —
# this cell only ingests and structurally validates the source.
# ============================================================

# Read every column as string with empty cells kept as "" (not NaN): this stops
# pandas coercing 'areaer_as_of' or the 'YYYY-MM' reclassification markers into
# datetimes, and keeps blank anchor/reclass cells as clean empty strings.
src = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)

# --- Structural schema check: columns are a FIXED transcription invariant -----
#     (this is a schema guard, not a vintage-specific value — safe to assert) ---
EXPECTED_COLS = ["country_name", "areaer_arrangement", "areaer_mpf",
                 "areaer_anchor_currency", "areaer_reclassified", "areaer_as_of"]
missing = [c for c in EXPECTED_COLS if c not in src.columns]
assert not missing, f"Source CSV missing expected columns: {missing}"   # fail-safe: stop on bad schema

# --- Report shape + arrangement breakdown for eyeballing ----------------------
#     Row count is deliberately NOT asserted: it grows if the IMF admits new
#     members, so hardcoding 195 would create a maintenance edit. Printed only. --
print("Loaded :", INPUT_CSV)
print("Shape  :", src.shape, "(rows, cols)")
print("Columns:", list(src.columns))
print("\nArrangement breakdown (should be the 10 IMF categories; counts sum to row total):")
print(src["areaer_arrangement"].value_counts().to_string())
print("\nSample rows:")
print(src.head(3).to_string(index=False))

Loaded : C:\Users\mjbou\governance-framework\data\raw\areaer_defacto_regime.csv
Shape  : (195, 6) (rows, cols)
Columns: ['country_name', 'areaer_arrangement', 'areaer_mpf', 'areaer_anchor_currency', 'areaer_reclassified', 'areaer_as_of']

Arrangement breakdown (should be the 10 IMF categories; counts sum to row total):
areaer_arrangement
Conventional peg                  41
Free floating                     32
Floating                          29
Stabilized arrangement            25
Crawl-like arrangement            25
No separate legal tender          15
Currency board                    12
Other managed arrangement         12
Crawling peg                       3
Pegged within horizontal bands     1

Sample rows:
    country_name       areaer_arrangement           areaer_mpf areaer_anchor_currency areaer_reclassified areaer_as_of
         Ecuador No separate legal tender exchange_rate_anchor                    usd                       2025-04-30
     El Salvador No separate legal ten

In [14]:
# ============================================================
# CELL 4 — DERIVE AS-OF DATE FROM DATA + ENCODE ORDINAL/GROUP
# (1) Reads the as-of date FROM the CSV's areaer_as_of column (never hardcoded),
#     asserting the matrix is a single-vintage snapshot.
# (2) Maps areaer_arrangement -> flexibility ordinal (1..10) and the IMF 4-way
#     group via the fixed Cell-2 lookups, with a structural guard that every
#     arrangement present in the data is covered (fails safe on a new/renamed
#     IMF category or a transcription typo — the ONLY place a taxonomy change
#     would surface, and it points you straight back to Cell 2).
# ============================================================

# --- (1) Derive the data-as-of date FROM the data, not from code --------------
# Appendix II.9 is a single-date snapshot, so areaer_as_of must be one unique
# value. Deriving it here means the annual refresh edits only the CSV; no date
# is ever typed into this notebook (the anti-"VDEM_VERSION" pattern).
as_of_values = src["areaer_as_of"].unique()
assert len(as_of_values) == 1, f"Expected ONE as-of date; found {list(as_of_values)}"
DATA_AS_OF = as_of_values[0]                    # e.g. '2025-04-30' — consumed by download_log in Cell 7

# --- (2) Structural vocabulary guard: every arrangement must be mappable -------
# Catches a new/renamed IMF category or a transcription typo BEFORE encoding.
# Vocabulary check, not a vintage-count check — safe to assert.
unmapped = sorted(set(src["areaer_arrangement"]) - set(ARRANGEMENT_ORDINAL))
assert not unmapped, f"Arrangement(s) missing from Cell-2 lookup (update ARRANGEMENT_ORDINAL/GROUP): {unmapped}"

# --- Apply the fixed methodology lookups to create typed columns --------------
# ordinal: 1 (most fixed) .. 10 (most flexible), per IMF matrix row order
# group  : hard_peg / soft_peg / floating / other_managed (IMF's own 4-way split;
#          other_managed is the RESIDUAL — treat its ordinal as provisional)
src["areaer_regime_ordinal"] = src["areaer_arrangement"].map(ARRANGEMENT_ORDINAL).astype(int)
src["areaer_regime_group"]   = src["areaer_arrangement"].map(ARRANGEMENT_GROUP)

# --- Verify before proceeding -------------------------------------------------
print("DATA_AS_OF (derived from CSV):", DATA_AS_OF)
print("\nOrdinal <-> arrangement (each arrangement must map to exactly one ordinal):")
print(src.groupby(["areaer_regime_ordinal", "areaer_arrangement"]).size().to_string())
print("\nGroup breakdown (other_managed = IMF residual):")
print(src["areaer_regime_group"].value_counts().to_string())

DATA_AS_OF (derived from CSV): 2025-04-30

Ordinal <-> arrangement (each arrangement must map to exactly one ordinal):
areaer_regime_ordinal  areaer_arrangement            
1                      No separate legal tender          15
2                      Currency board                    12
3                      Conventional peg                  41
4                      Stabilized arrangement            25
5                      Crawling peg                       3
6                      Crawl-like arrangement            25
7                      Pegged within horizontal bands     1
8                      Other managed arrangement         12
9                      Floating                          29
10                     Free floating                     32

Group breakdown (other_managed = IMF residual):
areaer_regime_group
soft_peg         95
floating         61
hard_peg         27
other_managed    12


In [15]:
# ============================================================
# CELL 5 — MAP JURISDICTIONS TO ISO3  (deterministic; NO fuzzy matching)
# Resolves each source country_name to an ISO 3166-1 alpha-3 code using ONLY
# (a) the explicit Cell-2 ISO3_OVERRIDES and (b) pycountry EXACT matches on the
# name / common_name / official_name fields. Fuzzy matching is deliberately
# avoided: this set contains fuzzy-dangerous pairs — Niger/Nigeria,
# Guinea / Guinea-Bissau / Equatorial Guinea, Congo / DR Congo, Sudan / South
# Sudan — where a fuzzy guess can silently mis-map. Any name that does not
# resolve deterministically is PRINTED and then hard-asserted, so the override
# list is completed from OBSERVED output (add the printed names to Cell-2
# overrides) rather than from my memory of pycountry's internal name coverage.
# ============================================================
import pycountry   # house-standard ISO library (already used by other pipelines)

def resolve_iso3(name):
    """Return (alpha_3, method); (None, 'UNRESOLVED') if no deterministic match."""
    # 1. explicit override — deterministic, takes precedence over everything
    if name in ISO3_OVERRIDES:
        return ISO3_OVERRIDES[name], "override"
    # 2. pycountry EXACT on the three name fields (no fuzzy — see cell header)
    for field in ("name", "common_name", "official_name"):
        try:
            m = pycountry.countries.get(**{field: name})
        except Exception:
            m = None
        if m is not None:
            return m.alpha_3, f"exact:{field}"
    # 3. no deterministic match — must be added to ISO3_OVERRIDES
    return None, "UNRESOLVED"

# Apply resolver; keep iso3 as a permanent column, method only as a local list
resolved = [resolve_iso3(n) for n in src["country_name"]]
src["iso3"] = [a for a, _ in resolved]
methods     = [m for _, m in resolved]

# --- Report: how each name resolved, plus any gaps/collisions -----------------
from collections import Counter
print("Resolution method breakdown:")
for method, n in sorted(Counter(methods).items()):
    print(f"  {method:<18} {n}")

# Names needing an override (empirically revealed — pycountry name coverage varies by version)
unresolved = [n for n, (a, _) in zip(src["country_name"], resolved) if a is None]
if unresolved:
    print("\n[ACTION] UNRESOLVED — add each to ISO3_OVERRIDES in Cell 2, then re-run Cell 2 + this cell:")
    for n in unresolved:
        print("   ", repr(n))

# Two source rows mapping to one ISO3 = a mis-map (each jurisdiction should be unique)
dups = [iso for iso, c in Counter([a for a in src["iso3"] if a]).items() if c > 1]
print("\nDuplicate ISO3 codes:", dups if dups else "none")

# --- Fail-safe: stop if anything is unresolved or duplicated ------------------
assert not unresolved, f"{len(unresolved)} unmapped name(s) — see list above; add to Cell-2 ISO3_OVERRIDES"
assert not dups, f"Duplicate ISO3 (likely a mis-map): {dups}"
print(f"\nAll {len(src)} jurisdictions resolved to a unique ISO3.")

Resolution method breakdown:
  exact:common_name  6
  exact:name         166
  exact:official_name 1
  override           22

Duplicate ISO3 codes: none

All 195 jurisdictions resolved to a unique ISO3.


In [18]:
# ============================================================
# CELL 6 — STRUCTURAL INTEGRITY GUARDS
# Validates the transformed frame against FIXED structural rules (value domains,
# null checks, internal consistency) — NOT against vintage-specific counts, so
# nothing here needs editing as the data updates. All violations are collected
# and reported together (diagnose the whole pattern, not just the first failure),
# then the cell halts before anything is written. Row TOTALS are printed, never
# asserted (they grow if the IMF admits new members).
# ============================================================
import re   # for validating the reclassified-date format

# --- Fixed AREAER vocabularies used only for domain checks --------------------
# ⚠️ MANUAL-MAINTENANCE CONSTANT: the AREAER monetary-policy-framework and anchor
#    vocabularies. Edit ONLY if the IMF revises its framework taxonomy (rare) — a
#    new value trips the domain checks below and points you here. (The arrangement/
#    ordinal/group vocabulary is already enforced via the Cell-2 lookups.)
VALID_MPF    = {"exchange_rate_anchor", "monetary_aggregate_target", "inflation_targeting", "other"}
VALID_ANCHOR = {"usd", "eur", "composite", "other"}   # set ONLY when mpf == exchange_rate_anchor

problems = []   # accumulate every violation so we see the full picture in one run

# --- (1) No null/blank in fields that will be scored or joined ----------------
for col in ["country_name", "iso3", "areaer_arrangement", "areaer_regime_ordinal",
            "areaer_regime_group", "areaer_mpf", "areaer_as_of"]:
    n_blank = int(src[col].isna().sum() + (src[col].astype(str).str.strip() == "").sum())
    if n_blank:
        problems.append(f"{col}: {n_blank} null/blank value(s)")

# --- (2) Domain checks: every value sits inside its allowed vocabulary ---------
bad_ord = set(src["areaer_regime_ordinal"]) - set(ARRANGEMENT_ORDINAL.values())
if bad_ord:    problems.append(f"ordinal outside 1..10: {sorted(bad_ord)}")
bad_grp = set(src["areaer_regime_group"]) - set(ARRANGEMENT_GROUP.values())
if bad_grp:    problems.append(f"group outside allowed set: {sorted(bad_grp)}")
bad_mpf = set(src["areaer_mpf"]) - VALID_MPF
if bad_mpf:    problems.append(f"mpf outside allowed set: {sorted(bad_mpf)}")
bad_anc = set(a for a in src["areaer_anchor_currency"] if a) - VALID_ANCHOR
if bad_anc:    problems.append(f"anchor_currency outside allowed set: {sorted(bad_anc)}")

# --- (3) Internal consistency: anchor present IFF mpf == exchange_rate_anchor --
extra_anchor = src[(src["areaer_mpf"] != "exchange_rate_anchor") & (src["areaer_anchor_currency"] != "")]
if len(extra_anchor):
    problems.append(f"{len(extra_anchor)} row(s) carry anchor_currency but mpf!=exchange_rate_anchor: {list(extra_anchor['country_name'])}")
missing_anchor = src[(src["areaer_mpf"] == "exchange_rate_anchor") & (src["areaer_anchor_currency"] == "")]
if len(missing_anchor):
    problems.append(f"{len(missing_anchor)} exchange_rate_anchor row(s) missing anchor_currency: {list(missing_anchor['country_name'])}")

# --- (4) ordinal<->group must be a consistent mapping (catches lookup drift) ---
grp_per_ord = src.groupby("areaer_regime_ordinal")["areaer_regime_group"].nunique()
if (grp_per_ord > 1).any():
    problems.append(f"ordinal(s) mapping to >1 group: {list(grp_per_ord[grp_per_ord>1].index)}")

# --- (5) reclassified must be blank or 'YYYY-MM' ------------------------------
bad_reclass = [(n, v) for n, v in zip(src["country_name"], src["areaer_reclassified"])
               if v != "" and not re.fullmatch(r"\d{4}-\d{2}", v)]
if bad_reclass:
    problems.append(f"malformed reclassified value(s): {bad_reclass}")

# --- Report all problems together, then fail-safe -----------------------------
if problems:
    print("INTEGRITY VIOLATIONS (nothing will be written):")
    for p in problems:
        print("  -", p)
    raise AssertionError(f"{len(problems)} integrity violation(s) — see above.")
else:
    print("All structural integrity guards PASSED.\n")
    print("Rows:", len(src))   # printed, not asserted (vintage-dependent)
    # Analytical sanity crosstab: confirms MPF distributes across groups as expected
    # (e.g. inflation_targeting appears only in soft_peg / floating / other_managed).
    print("\nGroup x MPF crosstab:")
    print(pd.crosstab(src["areaer_regime_group"], src["areaer_mpf"]).to_string())
    print("\nReclassified (regime-change recency) flags:", int((src["areaer_reclassified"] != "").sum()))

All structural integrity guards PASSED.

Rows: 195

Group x MPF crosstab:
areaer_mpf           exchange_rate_anchor  inflation_targeting  monetary_aggregate_target  other
areaer_regime_group                                                                             
floating                                0                   29                          7     25
hard_peg                               27                    0                          0      0
other_managed                           3                    1                          3      5
soft_peg                               52                   15                         16     12

Reclassified (regime-change recency) flags: 23


In [20]:
# ============================================================
# CELL 7 — WRITE CLEAN OUTPUT + REGISTER IN DOWNLOAD_LOG
# (1) Assembles the clean frame in the house CROSS-SECTION convention: country_code
#     (renamed from working 'iso3') + the AREAER fields. Matches rti_rating/pefa/
#     polfinance: code-keyed, NO 'year', NO 'country_name' (canonical name re-attached
#     downstream at merge). Adds areaer_as_of so the snapshot is self-describing.
# (2) Writes areaer_er_clean.csv to data/processed.
# (3) Registers the source in download_log with data_as_of_date pulled from DATA_AS_OF
#     (derived from the CSV in Cell 4) — NO date is ever typed into code here.
# NOTE: does NOT touch source_registry.csv — that is owned by notebook 02 (a separate
#       add-if-not-exists cell is provided after this one).
# ============================================================

# --- (1) Assemble clean output: rename working iso3 -> house 'country_code' ----
# Column order: identifier, arrangement + flexibility encoding, MPF detail, recency,
# vintage. country_name/year intentionally omitted (lean cross-section convention).
OUTPUT_COLS = ["country_code", "areaer_arrangement", "areaer_regime_ordinal",
               "areaer_regime_group", "areaer_mpf", "areaer_anchor_currency",
               "areaer_reclassified", "areaer_as_of"]
out = src.rename(columns={"iso3": "country_code"})[OUTPUT_COLS].copy()

# --- (2) Write the clean CSV to data/processed --------------------------------
out.to_csv(OUTPUT_CSV, index=False)
print("Wrote  :", OUTPUT_CSV)
print("Shape  :", out.shape, "(rows, cols)")
print("Columns:", list(out.columns))
print("\nSample:")
print(out.head(3).to_string(index=False))

# --- (3) Register in download_log; vintage is DATA_AS_OF (data-derived, not typed) -
# update_entry auto-stamps last_attempted_date. We set the refresh date to today,
# the data vintage to DATA_AS_OF, and build the version label FROM that same derived
# date, so nothing in this call needs manual editing on an annual refresh.
today_str = datetime.today().strftime("%Y-%m-%d")   # 'datetime' imported in Cell 0
update_entry(
    SOURCE_ID,
    last_successful_download_date=today_str,          # pipeline-run date (source is transcribed, not downloaded)
    data_as_of_date=DATA_AS_OF,                       # e.g. 2025-04-30 — derived from source CSV in Cell 4
    local_filename=os.path.basename(OUTPUT_CSV),      # areaer_er_clean.csv
    latest_available_version=f"AREAER de facto ER classification, as-of {DATA_AS_OF} (IMF Annual Report, Appendix II.9)",
    notes=(
        "De facto exchange-rate-regime classification (Concept 8, macro policy — PRIMARY tier-1; "
        "supersedes Reinhart-Rogoff de facto regime). HAND-TRANSCRIBED from the IMF Annual Report "
        "Appendix II.9 borderless matrix: AREAER Online is paywalled and the matrix PDF has no reliable "
        "automated extraction, so it is maintained as data/raw/areaer_defacto_regime.csv, validated at "
        "transcription against the PDF's per-category (row) AND per-column (MPF) country-count checksums. "
        "Fields: areaer_arrangement (10-way) + regime_ordinal (1-10 flexibility, matrix order) + regime_group "
        "(IMF 4-way; other_managed is a RESIDUAL, not a flexibility rank) + areaer_mpf (monetary-policy "
        "framework incl. inflation_targeting flag) + anchor_currency + reclassified (YYYY-MM regime-change "
        "recency). Cross-section snapshot; NO year (vintage = areaer_as_of / data_as_of_date). MANUAL REFRESH: "
        "re-transcribe reclassified countries + bump areaer_as_of in the source CSV, then re-run — NO code "
        "changes. ISO3 via pycountry + ISO3_OVERRIDES (add a mapping only if a new oddly-named jurisdiction "
        "appears; pipeline prints it). See instructions_data_maintenance.md."
    ),
)
print("\n--- download_log entry after update ---")
print_entry(SOURCE_ID)

Wrote  : C:\Users\mjbou\governance-framework\data\processed\areaer_er_clean.csv
Shape  : (195, 8) (rows, cols)
Columns: ['country_code', 'areaer_arrangement', 'areaer_regime_ordinal', 'areaer_regime_group', 'areaer_mpf', 'areaer_anchor_currency', 'areaer_reclassified', 'areaer_as_of']

Sample:
country_code       areaer_arrangement  areaer_regime_ordinal areaer_regime_group           areaer_mpf areaer_anchor_currency areaer_reclassified areaer_as_of
         ECU No separate legal tender                      1            hard_peg exchange_rate_anchor                    usd                       2025-04-30
         SLV No separate legal tender                      1            hard_peg exchange_rate_anchor                    usd                       2025-04-30
         MHL No separate legal tender                      1            hard_peg exchange_rate_anchor                    usd                       2025-04-30
[download_log] Updated entry for IMF_AREAER_ERREGIME

--- download_log en